# 2. Séquences — étiquetage BIO d'entités

**Objectif.** Assigner à chaque jeton `O`, `B-PER`, `B-ORG` ou `B-LOC`. `B` marque le début d'une entité ; les étiquettes `I-*` sont aussi prises en charge par l'évaluation des segments. Ce petit corpus français sert uniquement de démonstration hors ligne.

## 1. Phrases annotées

Le découpage se fait **par phrase**, afin qu'aucun jeton de la même phrase ne se retrouve à la fois dans l'apprentissage et le test.

In [1]:
from sklearn.model_selection import train_test_split

rows = [
    ("Alice travaille chez Atlas à Rabat", "B-PER O O B-ORG O B-LOC"),
    ("Karim étudie chez Nova à Fès", "B-PER O O B-ORG O B-LOC"),
    ("Salma visite Orion à Paris", "B-PER O B-ORG O B-LOC"),
    ("Nadia rejoint Atlas à Casablanca", "B-PER O B-ORG O B-LOC"),
    ("Youssef quitte Nova pour Tanger", "B-PER O B-ORG O B-LOC"),
    ("Amine rencontre Lina à Rabat", "B-PER O B-PER O B-LOC"),
    ("Sara rencontre Omar à Marrakech", "B-PER O B-PER O B-LOC"),
    ("Lina dirige Orion à Fès", "B-PER O B-ORG O B-LOC"),
    ("Omar représente Atlas à Paris", "B-PER O B-ORG O B-LOC"),
    ("Maya travaille pour Nova à Rabat", "B-PER O O B-ORG O B-LOC"),
    ("Atlas ouvre un bureau à Tanger", "B-ORG O O O O B-LOC"),
    ("Nova ouvre un bureau à Paris", "B-ORG O O O O B-LOC"),
    ("Orion recrute Alice à Casablanca", "B-ORG O B-PER O B-LOC"),
    ("Atlas recrute Karim à Marrakech", "B-ORG O B-PER O B-LOC"),
    ("Nova invite Salma à Fès", "B-ORG O B-PER O B-LOC"),
    ("Orion invite Nadia à Tanger", "B-ORG O B-PER O B-LOC"),
    ("Alice voyage de Rabat à Paris", "B-PER O O B-LOC O B-LOC"),
    ("Karim voyage de Fès à Marrakech", "B-PER O O B-LOC O B-LOC"),
    ("Salma voyage de Tanger à Rabat", "B-PER O O B-LOC O B-LOC"),
    ("Nadia voyage de Paris à Casablanca", "B-PER O O B-LOC O B-LOC"),
    ("Maya appelle Youssef depuis Fès", "B-PER O B-PER O B-LOC"),
    ("Omar appelle Amine depuis Rabat", "B-PER O B-PER O B-LOC"),
    ("Sara appelle Lina depuis Tanger", "B-PER O B-PER O B-LOC"),
    ("Atlas accueille Maya à Marrakech", "B-ORG O B-PER O B-LOC"),
    ("Nova accueille Omar à Casablanca", "B-ORG O B-PER O B-LOC"),
    ("Orion accueille Sara à Paris", "B-ORG O B-PER O B-LOC"),
    ("Youssef présente Nova à Amine", "B-PER O B-ORG O B-PER"),
    ("Maya présente Atlas à Sara", "B-PER O B-ORG O B-PER"),
    ("Lina présente Orion à Karim", "B-PER O B-ORG O B-PER"),
    ("Amine quitte Atlas pour Paris", "B-PER O B-ORG O B-LOC"),
]
sentences = [(text.split(), tags.split()) for text, tags in rows]
assert all(len(tokens) == len(tags) for tokens, tags in sentences)
train_sents, test_sents = train_test_split(sentences, test_size=0.3, random_state=42)
print(f"{len(sentences)} phrases ; train={len(train_sents)}, test={len(test_sents)}")
print("Exemple :", sentences[0])

30 phrases ; train=21, test=9
Exemple : (['Alice', 'travaille', 'chez', 'Atlas', 'à', 'Rabat'], ['B-PER', 'O', 'O', 'B-ORG', 'O', 'B-LOC'])


## 2. Caractéristiques locales et classifieur

Le classifieur voit le jeton courant et ses voisins immédiats. C'est une référence simple d'étiquetage de séquence, sans contrainte globale sur les transitions BIO.

In [2]:
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression

def token_features(tokens, i):
    word = tokens[i]
    return {
        "word": word.lower(), "prefix": word[:2].lower(), "suffix": word[-3:].lower(),
        "is_title": word.istitle(),
        "prev": tokens[i-1].lower() if i else "<START>",
        "next": tokens[i+1].lower() if i + 1 < len(tokens) else "<END>",
    }

def flatten(sents):
    features = [token_features(tokens, i) for tokens, _ in sents for i in range(len(tokens))]
    tags = [tag for _, labels in sents for tag in labels]
    return features, tags

train_features, train_tags = flatten(train_sents)
test_features, test_tags = flatten(test_sents)
vectorizer = DictVectorizer()
X_train = vectorizer.fit_transform(train_features)
X_test = vectorizer.transform(test_features)
model = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)
model.fit(X_train, train_tags)
pred_tags = model.predict(X_test).tolist()
print("Jetons train/test :", len(train_tags), len(test_tags))

Jetons train/test : 112 47


## 3. Métriques par jeton et par entité

L'exactitude par jeton inclut les nombreux `O`. Le F1 **par entité** exige le bon type et les bonnes bornes du segment ; c'est la métrique principale.

In [3]:
from sklearn.metrics import accuracy_score, f1_score, classification_report

def split_by_sentence(flat_tags, sents):
    result, offset = [], 0
    for tokens, _ in sents:
        result.append(flat_tags[offset:offset + len(tokens)])
        offset += len(tokens)
    assert offset == len(flat_tags)
    return result

def spans(tags):
    found, active_type, start = set(), None, None
    for i, tag in enumerate(list(tags) + ["O"]):
        if tag == "O":
            kind, entity_type = "O", None
        else:
            kind, entity_type = tag.split("-", 1)
        if active_type is not None and (kind != "I" or entity_type != active_type):
            found.add((start, i, active_type))
            active_type, start = None, None
        if kind == "B" or (kind == "I" and active_type is None):
            active_type, start = entity_type, i
    return found

def entity_scores(gold_sequences, predicted_sequences):
    tp = gold_count = pred_count = 0
    for gold, pred in zip(gold_sequences, predicted_sequences):
        gs, ps = spans(gold), spans(pred)
        tp += len(gs & ps)
        gold_count += len(gs)
        pred_count += len(ps)
    precision = tp / pred_count if pred_count else 0.0
    recall = tp / gold_count if gold_count else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {"precision": round(precision, 3), "recall": round(recall, 3),
            "f1": round(f1, 3), "correct": tp, "gold": gold_count, "predicted": pred_count}

gold_sequences = [tags for _, tags in test_sents]
pred_sequences = split_by_sentence(pred_tags, test_sents)
baseline_sequences = [["O"] * len(tokens) for tokens, _ in test_sents]
print("Exactitude jeton :", round(accuracy_score(test_tags, pred_tags), 3))
entity_labels = sorted(set(train_tags) - {"O"})
print("F1 macro des étiquettes d'entité :", round(f1_score(test_tags, pred_tags, labels=entity_labels, average="macro", zero_division=0), 3))
print("Entités — référence tout O :", entity_scores(gold_sequences, baseline_sequences))
print("Entités — modèle          :", entity_scores(gold_sequences, pred_sequences))
print(classification_report(test_tags, pred_tags, labels=entity_labels, zero_division=0))

Exactitude jeton : 0.957
F1 macro des étiquettes d'entité : 0.93
Entités — référence tout O : {'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'correct': 0, 'gold': 27, 'predicted': 0}
Entités — modèle          : {'precision': 0.926, 'recall': 0.926, 'f1': 0.926, 'correct': 25, 'gold': 27, 'predicted': 27}
              precision    recall  f1-score   support

       B-LOC       0.80      1.00      0.89         8
       B-ORG       1.00      1.00      1.00         8
       B-PER       1.00      0.82      0.90        11

   micro avg       0.93      0.93      0.93        27
   macro avg       0.93      0.94      0.93        27
weighted avg       0.94      0.93      0.93        27



## 4. Inspection de prédictions

Les erreurs de frontière ou de type restent visibles, même si l'exactitude par jeton est élevée.

In [4]:
for (tokens, gold), pred in list(zip(test_sents, pred_sequences))[:4]:
    print("Phrase :", " ".join(tokens))
    print("Vrai   :", gold)
    print("Prédit :", pred)
    print()

Phrase : Maya présente Atlas à Sara
Vrai   : ['B-PER', 'O', 'B-ORG', 'O', 'B-PER']
Prédit : ['B-PER', 'O', 'B-ORG', 'O', 'B-LOC']

Phrase : Orion invite Nadia à Tanger
Vrai   : ['B-ORG', 'O', 'B-PER', 'O', 'B-LOC']
Prédit : ['B-ORG', 'O', 'B-PER', 'O', 'B-LOC']

Phrase : Atlas accueille Maya à Marrakech
Vrai   : ['B-ORG', 'O', 'B-PER', 'O', 'B-LOC']
Prédit : ['B-ORG', 'O', 'B-PER', 'O', 'B-LOC']

Phrase : Karim voyage de Fès à Marrakech
Vrai   : ['B-PER', 'O', 'O', 'B-LOC', 'O', 'B-LOC']
Prédit : ['B-PER', 'O', 'O', 'B-LOC', 'O', 'B-LOC']



## 5. Limites

Noms et tournures sont répétés dans un corpus de 30 phrases. Le modèle ne garantit pas des transitions BIO valides et ne prouve aucune généralisation à des textes réels. Pour un projet réel, utiliser un corpus NER externe, des entités multi-jetons et une évaluation séparée par document.